<a href="https://colab.research.google.com/github/kavyasudha12/Gen-AI-Training-Task/blob/main/VIDEO_TEXT_SUMMARY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# STEP 1 — Install Requirements
!apt update -y
!apt install ffmpeg -y
!pip install -U openai faiss-cpu langchain tiktoken yt-dlp

In [ ]:
# STEP 2 — Load Secure API Key
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
# STEP 3 — Imports
!pip install langchain.text_splitters
from openai import OpenAI
from google.colab import files
from langchain_text_splitters import CharacterTextSplitter
import numpy as np
import faiss
import yt_dlp

client = OpenAI()

In [ ]:
# STEP 4 — Upload Video OR YouTube URL (CLEAN STATE)
import os

for file in ["youtube_video.mp4", "audio.wav"]:
    if os.path.exists(file):
        os.remove(file)
for var in ["transcript", "english_text", "original_language"]:
    if var in globals():
        del globals()[var]

print("✅ Cleared old video, audio, and data")

choice = input("Type '1' to Upload Video OR '2' for YouTube URL: ")

if choice == "1":
    uploaded = files.upload()
    video_path = list(uploaded.keys())[0]
    print("✅ Uploaded video:", video_path)

elif choice == "2":
    url = input("Paste YouTube URL: ")

    ydl_opts = {
        'format': 'mp4',
        'outtmpl': 'youtube_video.%(ext)s',
        'extractor_args': {
            'youtube': {'player_client': ['android']}
        },
        'quiet': True
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([url])

    video_path = "youtube_video.mp4"
    print("✅ Downloaded new YouTube video")

else:
    raise ValueError("Invalid choice")

In [ ]:
# STEP 5 — Extract Audio (FFmpeg)
audio_path = "audio.wav"

!ffmpeg -y -i "{video_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 {audio_path}

print("✅ Audio extracted")


In [ ]:
# STEP 6 — Speech to Text
with open(audio_path, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="gpt-4o-transcribe",
        file=audio_file
    )

transcript = transcription.text

print("\n===== TRANSCRIPT =====\n")
print(transcript)

In [ ]:
# STEP 7 — Detect Language
language_detection = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Identify the language. Reply with only the language name."},
        {"role": "user", "content": transcript[:2000]}
    ]
)

original_language = language_detection.choices[0].message.content.strip()
print("✅ Detected Language:", original_language)

# STEP 7B — Translate to English
translation = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Translate any language into clear English."},
        {"role": "user", "content": transcript}
    ]
)

english_text = translation.choices[0].message.content

print("\n===== ENGLISH TEXT =====\n")
print(english_text)

In [ ]:
# STEP 8 — Generate Summary
summary = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Summarize the video clearly."},
        {"role": "user", "content": english_text}
    ]
)

video_summary = summary.choices[0].message.content

print("\n===== VIDEO SUMMARY =====\n")
print(video_summary)


# STEP 8B — Summary in Original Language
summary_original = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Summarize the video clearly using the SAME language as the transcript."
        },
        {
            "role": "user",
            "content": transcript
        }
    ]
)

video_summary_original = summary_original.choices[0].message.content

print("\n===== VIDEO SUMMARY (ORIGINAL LANGUAGE) =====\n")
print(video_summary_original)

In [ ]:
# STEP 9 — Create Vector Database
splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = splitter.split_text(english_text)

embeddings = []

for chunk in texts:
    emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=chunk
    )
    embeddings.append(emb.data[0].embedding)

dimension = len(embeddings[0])
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))

print("✅ Vector Database Ready")

In [ ]:
# STEP 10 — Ask Questions
def ask_video(question):

    q_emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    ).data[0].embedding

    D, I = index.search(
        np.array([q_emb]).astype("float32"),
        k=3
    )

    context = " ".join([texts[i] for i in I[0]])

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer ONLY using the video content."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ]
    )

    return response.choices[0].message.content

In [ ]:
# STEP 11 — Example Queries
print("\n===== SAMPLE QUESTIONS =====")
print("Q1:", ask_video("What is the main topic?"))
print("Q2:", ask_video("Explain the key points discussed"))

In [ ]:
# STEP 13 — Interactive Q&A with Gradio
!pip install gradio -q

import gradio as gr


In [ ]:
def ask_video_bilingual(question):
    # Step 1: English answer (grounded in RAG context)
    q_emb = client.embeddings.create(
        model="text-embedding-3-small",
        input=question
    ).data[0].embedding

    D, I = index.search(
        np.array([q_emb]).astype("float32"),
        k=3
    )

    context = " ".join([texts[i] for i in I[0]])

    answer_en = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer ONLY using the given context.\n"
                    "If the answer is not in the context, reply exactly:\n"
                    "'❌ This question is external and not covered in the video.'"
                )
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}"
            }
        ]
    ).choices[0].message.content

    # If external, return as-is
    if "❌" in answer_en:
        return answer_en

    # Step 2: Translate answer back to original language

    answer_original = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": f"Translate the following answer into {original_language}."
        },
        {
            "role": "user",
            "content": answer_en
        }
    ]
).choices[0].message.content

    # Step 3: Return bilingual output
    return (
        "✅ **Answer (English):**\n"
        f"{answer_en}\n\n"
        "✅ **Answer (Original Language):**\n"
        f"{answer_original}"
    )

In [ ]:
import gradio as gr

def chat_response(user_message, chat_history):
    if not user_message.strip():
        return chat_history, ""

    answer = ask_video_bilingual(user_message)
    chat_history.append((user_message, answer))
    return chat_history, ""

def clear_chat():
    return []

with gr.Blocks() as demo:
    gr.Markdown(
        """
        # 🎬 Video Chat Assistant
        Ask questions about the video.

        """
    )

    chatbot = gr.Chatbot(
        label="Video Conversation",
        height=400
    )

    state = gr.State([])

    user_input = gr.Textbox(
        placeholder="Ask anything about the video and press Enter...",
        show_label=False
    )

    with gr.Row():
        clear_btn = gr.Button("🗑️ Clear Chat")
        exit_btn = gr.Button("❌ Exit")

    # Press ENTER to send
    user_input.submit(
        chat_response,
        inputs=[user_input, state],
        outputs=[chatbot, user_input]
    )

    clear_btn.click(
        clear_chat,
        outputs=chatbot
    )

    exit_btn.click(
        lambda: None,
        None,
        None
    )

demo.launch(share=True)